# 04 · Verifica: l'Hub accetta il certificato?
*Blocco 2 · IoT Hub con ADR e DPS. Passo di verifica (solo letture).*

`robot-01` ha il suo certificato. Tre domande: ADR lo conosce? L'Hub lo accetta? Chi lo ha emesso?

In [ ]:
# Carica la libreria e legge lo scenario (notebooks\_scenario.psd1): da qui in poi si usa $cfg
. ([scriptblock]::Create([IO.File]::ReadAllText((Get-ChildItem . , .\notebooks , .. -Filter _start.ps1 -ErrorAction SilentlyContinue | Select-Object -First 1).FullName)))

In [ ]:
if (-not $cfg -or -not $out) { throw "Mancano le variabili dello scenario: esegui prima la cella di caricamento (la prima cella di codice) di questo notebook." }
$id = 'robot-01'
$dir = (Get-ChildItem (Join-Path $cfg.Out "devices\$id") -Directory | Sort-Object Name | Select-Object -Last 1).FullName
$hub = ((Get-Content (Join-Path $dir 'dps-registration.json') -Raw | ConvertFrom-Json).registrationState).assignedHub
"$id sull'Hub $hub"
"cartella: $dir"

## ADR conosce il device
Il namespace ADR è il catalogo dei device: deve esserci, ed essere abilitato.

In [ ]:
"stato in ADR: " + (Get-AdrDeviceState -Name $id -Namespace $cfg.AdrName -ResourceGroup $cfg.Rg)

## L'Hub lo accetta
Un messaggio di prova, inviato all'Hub **con il certificato del device**. `204` vuol dire accettato, `401` rifiutato.

In [ ]:
Show-HubAcceptance -Label "$id con il suo certificato" -Hub $hub -DeviceId $id -StateDir $dir

## Chi lo ha emesso
La catena è: certificato del device, Issuing CA, Root CA, tutte nel namespace.

In [ ]:
Get-CertSummary (Join-Path $dir 'issued-chain.pem') | Format-List | Out-String

Prossimo notebook: **05**, i quattro modi di autenticare un device.